1. Cài đặt và import thư viện

In [ ]:
# Nạp thuật toán Random Forest Regression
from sklearn.ensemble import RandomForestRegressor

# Nạp các hàm đánh giá mô hình hồi quy
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Nạp hàm chia dữ liệu
from sklearn.model_selection import train_test_split

# Nạp thư viện tính toán
import numpy as np

# Nạp thư viện đo thời gian
import time

# Khai báo các giá trị số lượng cây cần khảo sát
n_estimators_values = [50, 100, 150, 200]

# Khai báo các giá trị độ sâu tối đa của cây
max_depth_values = [10, 15, 20, None]

# Khai báo số mẫu tối thiểu để chia một nút
min_samples_split_values = [2, 5, 10]

# Khai báo số mẫu tối thiểu tại nút lá
min_samples_leaf_values = [1, 2, 5]

# Tính tổng số tổ hợp tham số
total_configs = (
    len(n_estimators_values)
    * len(max_depth_values)
    * len(min_samples_split_values)
    * len(min_samples_leaf_values)
)

print("Tổng số tổ hợp tham số:", total_configs)

Tổng số tổ hợp tham số: 144


2. Đọc bộ dữ liệu Diamonds Prices 2022

In [ ]:
import kagglehub
import os
import pandas as pd
import numpy as np

# Tải dữ liệu
path = kagglehub.dataset_download("nancyalaswad90/diamonds-prices")

# Đường dẫn đến file CSV
file_path = os.path.join(path, "Diamonds Prices2022.csv")

# Đọc dữ liệu
diamonds = pd.read_csv(file_path)

print("Kích thước dữ liệu ban đầu:", diamonds.shape)
print(diamonds.head())

Using Colab cache for faster access to the 'diamonds-prices' dataset.
Kích thước dữ liệu ban đầu: (53943, 11)
   Unnamed: 0  carat      cut color clarity  depth  table  price     x     y  \
0           1   0.23    Ideal     E     SI2   61.5   55.0    326  3.95  3.98   
1           2   0.21  Premium     E     SI1   59.8   61.0    326  3.89  3.84   
2           3   0.23     Good     E     VS1   56.9   65.0    327  4.05  4.07   
3           4   0.29  Premium     I     VS2   62.4   58.0    334  4.20  4.23   
4           5   0.31     Good     J     SI2   63.3   58.0    335  4.34  4.35   

      z  
0  2.43  
1  2.31  
2  2.31  
3  2.63  
4  2.75  


3. Khám phá và kiểm tra dữ liệu ban đầu

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

# Xóa cột không cần thiết
if "Unnamed: 0" in diamonds.columns:
    diamonds = diamonds.drop(columns=["Unnamed: 0"])

# Loại bỏ các mẫu có kích thước không hợp lệ
before = len(diamonds)

diamonds = diamonds[
    (diamonds["x"] > 0) &
    (diamonds["y"] > 0) &
    (diamonds["z"] > 0)
].copy()

print("Số dòng kích thước không hợp lệ:", before - len(diamonds))

# Xóa các dòng trùng lặp
duplicates = diamonds.duplicated().sum()
diamonds = diamonds.drop_duplicates()

print("Số dòng trùng lặp:", duplicates)

# Thứ tự mã hóa các thuộc tính
cut_categories = ["Fair", "Good", "Very Good", "Premium", "Ideal"]
color_categories = ["J", "I", "H", "G", "F", "E", "D"]
clarity_categories = ["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"]

# Khởi tạo bộ mã hóa
encoder = OrdinalEncoder(
    categories=[
        cut_categories,
        color_categories,
        clarity_categories
    ]
)

# Mã hóa các thuộc tính phân loại
diamonds[["cut", "color", "clarity"]] = encoder.fit_transform(
    diamonds[["cut", "color", "clarity"]]
)

print("Kích thước sau tiền xử lý:", diamonds.shape)
print(diamonds.head())

Số dòng kích thước không hợp lệ: 20
Số dòng trùng lặp: 148
Kích thước sau tiền xử lý: (53775, 10)
   carat  cut  color  clarity  depth  table  price     x     y     z
0   0.23  4.0    5.0      1.0   61.5   55.0    326  3.95  3.98  2.43
1   0.21  3.0    5.0      2.0   59.8   61.0    326  3.89  3.84  2.31
2   0.23  1.0    5.0      4.0   56.9   65.0    327  4.05  4.07  2.31
3   0.29  3.0    1.0      3.0   62.4   58.0    334  4.20  4.23  2.63
4   0.31  1.0    0.0      1.0   63.3   58.0    335  4.34  4.35  2.75


4. Tiền xử lý dữ liệu

In [ ]:
from sklearn.model_selection import train_test_split

# Tách các thuộc tính đầu vào
X = diamonds.drop(columns=["price"])

# Tách thuộc tính mục tiêu là giá kim cương
y = diamonds["price"]

# Chia dữ liệu thành 80% Train và 20% Test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Hiển thị số lượng mẫu
print("Số mẫu Train:", len(X_train))
print("Số mẫu Test:", len(X_test))

Số mẫu Train: 43020
Số mẫu Test: 10755


5. Mã hóa các thuộc tính phân loại

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import time

# Các giá trị số lượng cây cần thử
n_estimators_values = [50, 100, 150, 200]

# Các giá trị độ sâu tối đa
max_depth_values = [10, 15, 20, None]

# Số mẫu tối thiểu để chia một nút
min_samples_split_values = [2, 5, 10]

# Số mẫu tối thiểu tại một nút lá
min_samples_leaf_values = [1, 2, 5]

# Tính tổng số tổ hợp tham số
total_configs = (
    len(n_estimators_values)
    * len(max_depth_values)
    * len(min_samples_split_values)
    * len(min_samples_leaf_values)
)

print("Tổng số tổ hợp tham số:", total_configs)

Tổng số tổ hợp tham số: 144


Chia dữ liệu Hold-out (80% Train – 20% Test)

In [ ]:
# Chia tập Train thành Train nhỏ và Validation
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42
)

# Hiển thị số lượng mẫu
print("Số mẫu Train dùng để khảo sát:", len(X_tr))
print("Số mẫu Validation:", len(X_val))
print("Số mẫu Test giữ riêng:", len(X_test))

Số mẫu Train dùng để khảo sát: 34416
Số mẫu Validation: 8604
Số mẫu Test giữ riêng: 10755


8. Khảo sát và lựa chọn siêu tham số Random Forest

In [ ]:
# Khởi tạo giá trị RMSE tốt nhất
best_rmse = float("inf")

# Lưu bộ tham số tốt nhất
best_params = None

# Lưu thời gian huấn luyện của cấu hình tốt nhất
best_time = None

# Danh sách lưu kết quả tất cả cấu hình
all_results = []

# Đếm số cấu hình đã chạy
count = 0

# Duyệt các giá trị số lượng cây
for n_estimators in n_estimators_values:

    # Duyệt các giá trị độ sâu
    for max_depth in max_depth_values:

        # Duyệt các giá trị số mẫu tối thiểu để chia nút
        for min_samples_split in min_samples_split_values:

            # Duyệt các giá trị số mẫu tối thiểu tại nút lá
            for min_samples_leaf in min_samples_leaf_values:

                # Tạo mô hình Random Forest
                model = RandomForestRegressor(
                    n_estimators=n_estimators,
                    max_depth=max_depth,
                    min_samples_split=min_samples_split,
                    min_samples_leaf=min_samples_leaf,
                    criterion="squared_error",
                    random_state=42,
                    n_jobs=-1
                )

                # Bắt đầu tính thời gian
                start_time = time.perf_counter()

                # Huấn luyện mô hình
                model.fit(X_tr, y_tr)

                # Tính thời gian huấn luyện
                train_time = time.perf_counter() - start_time

                # Dự đoán trên Validation
                y_pred_val = model.predict(X_val)

                # Tính các chỉ số đánh giá
                mae = mean_absolute_error(y_val, y_pred_val)
                mse = mean_squared_error(y_val, y_pred_val)
                rmse = np.sqrt(mse)
                r2 = r2_score(y_val, y_pred_val)

                # Lưu kết quả cấu hình hiện tại
                all_results.append({
                    "n_estimators": n_estimators,
                    "max_depth": max_depth,
                    "min_samples_split": min_samples_split,
                    "min_samples_leaf": min_samples_leaf,
                    "MAE": mae,
                    "MSE": mse,
                    "RMSE": rmse,
                    "R2": r2,
                    "Thoi_gian": train_time
                })

                # Cập nhật nếu RMSE tốt hơn
                if rmse < best_rmse:
                    best_rmse = rmse

                    best_params = {
                        "n_estimators": n_estimators,
                        "max_depth": max_depth,
                        "min_samples_split": min_samples_split,
                        "min_samples_leaf": min_samples_leaf
                    }

                    best_time = train_time

                # Tăng số cấu hình đã chạy
                count += 1

                # In tiến độ sau mỗi 10 cấu hình
                if count % 10 == 0 or count == total_configs:
                    print(f"Đã chạy {count}/{total_configs} cấu hình")

# Hiển thị kết quả
print("\nBỘ THAM SỐ TỐT NHẤT")
print(best_params)

print(f"RMSE tốt nhất trên Validation: {best_rmse:.4f}")
print(f"Thời gian huấn luyện cấu hình tốt nhất: {best_time:.4f} giây")

Đã chạy 10/144 cấu hình
Đã chạy 20/144 cấu hình
Đã chạy 30/144 cấu hình
Đã chạy 40/144 cấu hình
Đã chạy 50/144 cấu hình
Đã chạy 60/144 cấu hình
Đã chạy 70/144 cấu hình
Đã chạy 80/144 cấu hình
Đã chạy 90/144 cấu hình
Đã chạy 100/144 cấu hình
Đã chạy 110/144 cấu hình
Đã chạy 120/144 cấu hình
Đã chạy 130/144 cấu hình
Đã chạy 140/144 cấu hình
Đã chạy 144/144 cấu hình

BỘ THAM SỐ TỐT NHẤT
{'n_estimators': 150, 'max_depth': 15, 'min_samples_split': 2, 'min_samples_leaf': 2}
RMSE tốt nhất trên Validation: 564.6077
Thời gian huấn luyện cấu hình tốt nhất: 13.8212 giây


9. Huấn luyện Random Forest với bộ tham số tốt nhất

In [ ]:
from sklearn.ensemble import RandomForestRegressor
import time

# Tạo mô hình với bộ tham số tốt nhất
best_model = RandomForestRegressor(
    n_estimators=150,
    max_depth=15,
    min_samples_split=2,
    min_samples_leaf=2,
    criterion="squared_error",
    random_state=42,
    n_jobs=-1
)

# Bắt đầu đo thời gian
start_time = time.perf_counter()

# Huấn luyện lại trên toàn bộ tập Train
best_model.fit(X_train, y_train)

# Tính thời gian huấn luyện cuối cùng
final_train_time = time.perf_counter() - start_time

print("Đã huấn luyện Random Forest thành công!")
print(f"Thời gian huấn luyện: {final_train_time:.4f} giây")

Đã huấn luyện Random Forest thành công!
Thời gian huấn luyện: 17.5786 giây


10. Đánh giá kết quả Hold-out trên Train và Test

In [ ]:
# Dự đoán giá kim cương trên tập Train
y_pred_train = best_model.predict(X_train)

# Tính MAE
mae_train = mean_absolute_error(y_train, y_pred_train)

# Tính MSE
mse_train = mean_squared_error(y_train, y_pred_train)

# Tính RMSE
rmse_train = np.sqrt(mse_train)

# Tính R bình phương
r2_train = r2_score(y_train, y_pred_train)

# Hiển thị kết quả
print("KẾT QUẢ TRÊN TẬP TRAIN")
print(f"MAE: {mae_train:.4f}")
print(f"MSE: {mse_train:.4f}")
print(f"RMSE: {rmse_train:.4f}")
print(f"R²: {r2_train:.4f}")

KẾT QUẢ TRÊN TẬP TRAIN
MAE: 161.7726
MSE: 102527.2928
RMSE: 320.1988
R²: 0.9936


In [ ]:
# Dự đoán giá kim cương trên tập Test
y_pred_test = best_model.predict(X_test)

# Tính MAE
mae_test = mean_absolute_error(y_test, y_pred_test)

# Tính MSE
mse_test = mean_squared_error(y_test, y_pred_test)

# Tính RMSE
rmse_test = np.sqrt(mse_test)

# Tính R bình phương
r2_test = r2_score(y_test, y_pred_test)

# Hiển thị kết quả
print("KẾT QUẢ TRÊN TẬP TEST")
print(f"MAE: {mae_test:.4f}")
print(f"MSE: {mse_test:.4f}")
print(f"RMSE: {rmse_test:.4f}")
print(f"R²: {r2_test:.4f}")

KẾT QUẢ TRÊN TẬP TEST
MAE: 266.6035
MSE: 270082.0118
RMSE: 519.6942
R²: 0.9829


In [ ]:
import pandas as pd
from IPython.display import display

# Bảng bộ tham số tốt nhất
params_table = pd.DataFrame({
    "Tham số": [
        "n_estimators",
        "max_depth",
        "min_samples_split",
        "min_samples_leaf",
        "Thời gian huấn luyện mô hình"
    ],
    "Giá trị": [
        best_params["n_estimators"],
        best_params["max_depth"],
        best_params["min_samples_split"],
        best_params["min_samples_leaf"],
        f"{final_train_time:.4f} giây"
    ]
})

# Bảng kết quả Train và Test
results_table = pd.DataFrame({
    "Tập dữ liệu": ["Train", "Test"],
    "MAE": [mae_train, mae_test],
    "MSE": [mse_train, mse_test],
    "RMSE": [rmse_train, rmse_test],
    "R²": [r2_train, r2_test]
})

# Hiển thị bảng tham số
print("BỘ THAM SỐ CHO KẾT QUẢ TỐT NHẤT")
display(params_table)

# Hiển thị bảng kết quả
print("\nKẾT QUẢ HUẤN LUYỆN TƯƠNG ỨNG")
display(results_table.round(4))

BỘ THAM SỐ CHO KẾT QUẢ TỐT NHẤT


,Tham số,Giá trị
0,n_estimators,150
1,max_depth,15
2,min_samples_split,2
3,min_samples_leaf,2
4,Thời gian huấn luyện mô hình,17.5786 giây



KẾT QUẢ HUẤN LUYỆN TƯƠNG ỨNG


,Tập dữ liệu,MAE,MSE,RMSE,R²
0,Train,161.7726,102527.2928,320.1988,0.9936
1,Test,266.6035,270082.0118,519.6942,0.9829
